# Unsolved Networks Analysis

In [1]:
import pypsa
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import os

### Choose the networks to analyse

In [17]:
capacity_scenarios = ["germany_scenario1", "germany_scenario2"]

In [18]:
home = "/home/lucakristin/Desktop/my_pypsa"
cluster = "450"
folder = "germany_base2_"

### Create folder and paths

In [19]:
# Create analysis output directory and logging file
analysis_dir = f"{home}/pypsa-eur/analysis/{folder}"
os.makedirs(analysis_dir, exist_ok=True)
log_file = os.path.join(analysis_dir, "summary.txt")

# Initialize/clear the log file with header
with open(log_file, "w") as _f:
    _f.write(f"Networks comparison summary\n")

def log_print(*args, sep=' ', end='\n', also_print=True):
    """Helper to print and append to the summary log"""
    s = sep.join(str(a) for a in args) + end
    if also_print:
        print(*args, sep=sep, end=end)
    with open(log_file, 'a') as _f:
        _f.write(s)

def save_fig(fig, name, dpi=150):
    """Helper to save a matplotlib figure into the analysis folder"""
    fname = os.path.join(analysis_dir, name)
    fig.savefig(fname, bbox_inches='tight', dpi=dpi)
    log_print(f"Saved figure: {fname}")

### Set paths

In [20]:
# Paths to the three networks
network_paths = {
    "windy": f"{home}/pypsa-eur/resources/{folder}windy/networks/base_s_{cluster}_elec_.nc",
    "notwindy": f"{home}/pypsa-eur/resources/{folder}notwindy/networks/base_s_{cluster}_elec_.nc",
    "windvariability": f"{home}/pypsa-eur/resources/{folder}windvariability/networks/base_s_{cluster}_elec_.nc"
}

# Load the networks
networks = {name: pypsa.Network(path) for name, path in network_paths.items()}
n1 = networks["windy"]
n2 = networks["notwindy"]
n3 = networks["windvariability"]

n1.name = "Windy"
n2.name = "Not Windy"
n3.name = "Wind Variability"

INFO:pypsa.network.io:New version 1.2.0 available! (Current: 1.1.2)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.2.0 available! (Current: 1.1.2)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.2.0 available! (Current: 1.1.2)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks


### Check

### Components

In [21]:
log_print("\n=== Network Statistics ===")
for n in networks.values():
    log_print(f"\nNetwork: {n.name}")
    for dfname in ["buses","lines","generators","loads","links","storage_units","stores"]:
        if hasattr(n, dfname):
            log_print(f"  {dfname}: {len(getattr(n, dfname))}")


=== Network Statistics ===

Network: Windy
  buses: 1350
  lines: 628
  generators: 3242
  loads: 450
  links: 1800
  storage_units: 60
  stores: 900

Network: Not Windy
  buses: 1350
  lines: 628
  generators: 3242
  loads: 450
  links: 1800
  storage_units: 60
  stores: 900

Network: Wind Variability
  buses: 1350
  lines: 628
  generators: 3242
  loads: 450
  links: 1800
  storage_units: 60
  stores: 900


### Statistics

In [22]:
log_print("\n=== Network Statistics ===")
for n in networks.values():
    log_print(f"\nNetwork: {n.name}")
    log_print(n.statistics().to_string())
    log_print("\n")

log_print("\n=== Analysis Complete ===")
log_print(f"All results saved to: {analysis_dir}")


=== Network Statistics ===

Network: Windy
                                  Optimal Capacity  Installed Capacity  Supply  Withdrawal  Energy Balance Transmission Capacity Factor  Curtailment  Capital Expenditure  Operational Expenditure  Revenue  Market Value
Generator   Combined-Cycle Gas                 0.0        3.078313e+04     0.0         0.0             0.0          0.0             0.0          0.0                  0.0                      0.0      0.0           0.0
            Offshore Wind (AC)                 0.0        1.116374e+04     0.0         0.0             0.0          0.0             0.0          0.0                  0.0                      0.0      0.0           0.0
            Onshore Wind                       0.0        7.333207e+04     0.0         0.0             0.0          0.0             0.0          0.0                  0.0                      0.0      0.0           0.0
            Open-Cycle Gas                     0.0        6.112744e+03     0.0      

In [23]:
component_specs = [("generators", "p_nom"), ("storage_units", "p_nom"), ("links", "p_nom")]

for network_name, network in networks.items():
    carrier_capacity = pd.Series(dtype=float)

    for component_name, capacity_column in component_specs:
        component = getattr(network, component_name, None)
        if component is None or component.empty or "carrier" not in component.columns or capacity_column not in component.columns:
            continue
        carrier_capacity = carrier_capacity.add(component.groupby("carrier")[capacity_column].sum(), fill_value=0)

    carrier_capacity = carrier_capacity.sort_values(ascending=False)
    print(f"\n{network.name} installed capacity by carrier [MW]")
    if carrier_capacity.empty:
        print("  No carrier capacity found.")
    else:
        print(carrier_capacity.to_string(float_format=lambda value: f"{value:.2f}"))


Windy installed capacity by carrier [MW]
carrier
solar                86408.00
onwind               73332.07
CCGT                 30783.13
coal                 20351.94
lignite              19457.03
offwind-ac           11163.74
biomass               8020.00
PHS                   7361.72
OCGT                  6112.74
oil                   5684.35
ror                   4014.12
waste                 3127.66
hydro                  402.24
geothermal              26.90
H2 Fuel Cell             0.00
H2 Electrolysis          0.00
battery charger          0.00
battery discharger       0.00
offwind-float            0.00
offwind-dc               0.00
solar-hsat               0.00

Not Windy installed capacity by carrier [MW]
carrier
solar                86408.00
onwind               73332.07
CCGT                 30783.13
coal                 20351.94
lignite              19457.03
offwind-ac           11163.74
biomass               8020.00
PHS                   7361.72
OCGT                  6112